# Price model on city C (3.7, steps 7 and 8)
**City C = Santa Clara County**, the unseen new client. This notebook scores it with the **saved, unchanged model named in `MODEL_NAME`** (`price_model_v1` for the first run, `price_model_v2` after the change; no refit, no new features) and records what happens: did it run, what failed, what came out, and the MAE on C.

**Order.**
1. load the saved model and the data of city C (from the Silver table; CSV fallback if the ETL for C has not run yet);
2. check C against A/B: missing columns, feature drift, unseen categories, location, price level;
3. score C with the saved model and compare the three numbers: baseline on C, model MAE on A and B, model MAE on C;
4. see where the error comes from, collect observations, publish the C rows and result files.

**Everything after the first run (the change list, `price_model_v2`) is step 8 and is done in `src/price_model.py` and `price_model.ipynb`; see the last section.**

## 1. Setup

In [0]:
import datetime
import importlib
import os
import sys

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
assert os.path.exists(os.path.join(SRC_DIR, "price_model.py")), f"price_model.py not found in {SRC_DIR}"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import price_model as pm
importlib.reload(pm)

CATALOG, SCHEMA = "workspace", "airbnb_eda"                  # model-evaluation objects live here, as in price_model.ipynb
SOURCE_TABLE = "workspace.airbnb.silver_listings"            # output of the 3.3 pipeline (run it for C first: allow_city_c=true)
COLUMN_MAP = {}                                              # Silver name -> model name, only if the names differ from pm.REQUIRED_COLUMNS
CITY_C = "Santa_Clara_County"
TRAIN_CITIES = ["Los_Angeles", "San_Francisco"]
MODEL_NAME = "price_model_v2"                                # the saved model that is scored; never refitted here
PUBLISH = True                                               # write the C rows to gold_predictions and the C metrics (only own rows are replaced)
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))      # CSV fallback
MODEL_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/raw/models"
RESULTS_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "results"))
GOLD = {"predictions": "workspace.airbnb.gold_predictions",
        "metrics": f"{CATALOG}.{SCHEMA}.gold_price_model_metrics",
        "split": f"{CATALOG}.{SCHEMA}.price_model_split"}

OBSERVATIONS = []        # (area, text) collected while running: the raw material for the change list


def note(area, text):
    OBSERVATIONS.append((area, text))
    print(f"NOTE [{area}] {text}")


def table_exists(name):
    return spark.catalog.tableExists(name)


def read_table(name):
    return spark.table(name)


def sql_list(values):
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in sorted(set(values)))


def write_rows(sdf, table, replace_where):
    # replace only the rows that match replace_where; an existing table with other columns is left alone
    if table_exists(table):
        existing = set(read_table(table).columns)
        if existing != set(sdf.columns):
            raise ValueError(f"{table} exists with different columns, not touching it.\nexisting: {sorted(existing)}\nnew: {sorted(sdf.columns)}")
        sdf.write.mode("overwrite").option("replaceWhere", replace_where).saveAsTable(table)
        return False
    sdf.write.saveAsTable(table)
    return True

## 2. The saved model (`MODEL_NAME`)

**What is being scored.** One row per item: the model that was fitted on A and B, when it was saved, and its A/B test MAE.

In [0]:
model_path = f"{MODEL_DIR}/{MODEL_NAME}.joblib"
assert os.path.exists(model_path), f"{model_path} not found: this notebook scores with the saved model and never refits"
saved = pm.load_model(model_path)
model, baseline, baseline_stay = saved["model"], saved["baseline"], saved["baseline_stay_type"]
meta = saved["meta"]
fitted_columns = list(getattr(model.pipeline_, "feature_names_in_", pm.FEATURES))
display(pd.DataFrame({
    "item": ["model", "fitted on cities", "training listings", "features used", "saved at", "test MAE saved with the model"],
    "value": [meta.get("chosen_candidate"), ", ".join(meta.get("cities", [])), model.n_train_, len(fitted_columns),
              meta.get("trained_at"), str(meta.get("test_mae"))]}))

item,value
model,"boosting, 31 leaves"
fitted on cities,"Los_Angeles, San_Francisco"
training listings,33881
features used,18
saved at,2026-11-12 01:42:29
test MAE saved with the model,"{'A: Los_Angeles | baseline': 180.49, 'A: Los_Angeles | baseline + stay type': 171.6, 'A: Los_Angeles | model': 111.52, 'B: San_Francisco | baseline': 245.8, 'B: San_Francisco | baseline + stay type': 227.33, 'B: San_Francisco | model': 150.57, 'A+B: both | baseline': 189.25, 'A+B: both | baseline + stay type': 179.08, 'A+B: both | model': 116.75}"


## 3. Load city C

The data of C comes from the Silver table (rows with `city = Santa_Clara_County`). If the table has no such rows yet, the CSV in `data/` is read instead and a note is made: the proper first run needs the ETL output.

In [0]:
def read_silver(cities):
    src = spark.table(SOURCE_TABLE)
    src = src.select([F.col(c).alias(COLUMN_MAP.get(c, c)) for c in src.columns])
    have = [c for c in pm.REQUIRED_COLUMNS + pm.OPTIONAL_COLUMNS if c in src.columns]
    absent = [c for c in pm.REQUIRED_COLUMNS if c not in src.columns]
    return src.filter(F.col("city").isin(list(cities))).select(have).toPandas(), absent


cities_in_silver = sorted(r[0] for r in spark.table(SOURCE_TABLE).select("city").distinct().collect())
print("cities in", SOURCE_TABLE, ":", cities_in_silver)

if CITY_C in cities_in_silver:
    raw_c, absent_columns = read_silver([CITY_C])
    source_used = f"Silver table {SOURCE_TABLE}"
else:
    raw_c = pm.load_listings_csv(DATA_DIR, [CITY_C])
    absent_columns = []
    source_used = "CSV in data/ (the Silver table has no rows for C)"
    note("pipeline", "the Silver table has no rows for city C: the ETL has not been run for C (allow_city_c=true), CSV used instead")
if absent_columns:
    note("pipeline", f"columns required by the model are missing in the Silver table: {absent_columns}")

c_data = pm.add_features(raw_c).reset_index(drop=True)
assert len(c_data) > 0, "no listings for city C"
c_data["price_status"] = pm.price_status(c_data)
ab = pm.add_features(read_silver(TRAIN_CITIES)[0]).reset_index(drop=True)
ab["price_status"] = pm.price_status(ab)
print("source of C:", source_used, "| listings in C:", len(c_data))

cities in workspace.airbnb.silver_listings : ['Los_Angeles', 'San_Francisco', 'Santa_Clara_County']
source of C: Silver table workspace.airbnb.silver_listings | listings in C: 6831


**Price status.** One row per city: all listings, with a valid price (can be scored), without any price, and placeholder prices.

In [0]:
def status_row(d, name):
    s = d["price_status"].value_counts()
    return {"city": name, "listings": len(d), "valid": int(s.get("valid", 0)), "no_price": int(s.get("no_price", 0)),
            "placeholder": int(s.get("placeholder", 0)), "valid_pct": round(100 * s.get("valid", 0) / len(d), 1)}

display(pd.DataFrame([status_row(ab[ab.city == c], c) for c in TRAIN_CITIES] + [status_row(c_data, CITY_C)]))
n_valid_c = int((c_data.price_status == "valid").sum())
assert n_valid_c > 0, "no listing of city C has a valid price: nothing to score against"

city,listings,valid,no_price,placeholder,valid_pct
Los_Angeles,43781,36676,7094,11,83.8
San_Francisco,7405,5676,1714,15,76.7
Santa_Clara_County,6831,5469,1355,7,80.1


## 4. Is C like A and B? (before looking at any error)

**Missing values in the model's columns.** One row per column: % missing in A/B and in C, and the difference. A column that is empty in C is something the model silently loses.

In [0]:
miss = pd.DataFrame({"pct_missing_AB": 100 * ab[pm.REQUIRED_COLUMNS].isna().mean(),
                     "pct_missing_C": 100 * c_data[pm.REQUIRED_COLUMNS].isna().mean()}).round(2)
miss["difference"] = (miss["pct_missing_C"] - miss["pct_missing_AB"]).round(2)
display(miss.rename_axis("column").reset_index().sort_values("difference", key=lambda s: s.abs(), ascending=False))
for col, r in miss.iterrows():
    if r.pct_missing_C == 100 and r.pct_missing_AB < 100:
        note("data", f"column {col} is empty in C (missing in {r.pct_missing_AB:.1f} % of A/B)")
    elif r["difference"] > 20:
        note("data", f"column {col} is missing far more often in C ({r.pct_missing_C:.1f} %) than in A/B ({r.pct_missing_AB:.1f} %)")

column,pct_missing_AB,pct_missing_C,difference
price,17.21,19.82,2.61
beds,2.69,2.52,-0.17
minimum_nights,0.06,0.22,0.16
bedrooms,0.18,0.25,0.07
bathroom_shared,0.18,0.16,-0.02
bathrooms_num,0.18,0.16,-0.02
host_is_superhost,0.0,0.0,0.0
latitude,0.0,0.0,0.0
has_license,0.0,0.0,0.0
hosts_time_as_host_years,0.0,0.0,0.0


**Numeric features.** One row per feature: mean and median in A/B and in C, and `standardized_difference` = (mean C - mean A/B) / std A/B (above 1 in absolute value = C is far from what the model saw).

In [0]:
rows = []
for f in pm.NUM_FEATURES:
    a, c = ab[f], c_data[f]
    sd = a.std()
    rows.append({"feature": f, "mean_AB": a.mean(), "mean_C": c.mean(), "median_AB": a.median(), "median_C": c.median(),
                 "standardized_difference": (c.mean() - a.mean()) / sd if sd and sd > 0 else np.nan,
                 "pct_missing_AB": 100 * a.isna().mean(), "pct_missing_C": 100 * c.isna().mean()})
drift = pd.DataFrame(rows).sort_values("standardized_difference", key=lambda s: s.abs(), ascending=False)
display(drift.round(2))
for _, r in drift[drift.standardized_difference.abs() > 1].iterrows():
    note("model", f"feature {r.feature} is far from A/B in C: mean {r.mean_C:.2f} against {r.mean_AB:.2f} (standardized difference {r.standardized_difference:.2f})")

feature,mean_AB,mean_C,median_AB,median_C,standardized_difference,pct_missing_AB,pct_missing_C
has_license,0.35,0.06,0.0,0.0,-0.6,0.0,0.0
is_monthly_rental,0.47,0.2,0.0,0.0,-0.54,0.06,0.22
dist_to_center_km,15.65,10.17,11.65,8.46,-0.39,0.0,0.0
bathroom_shared,0.1,0.22,0.0,0.0,0.37,0.18,0.16
minimum_nights_capped,17.85,9.8,5.0,2.0,-0.26,0.06,0.22
accommodates,4.04,3.45,3.0,2.0,-0.21,0.0,0.0
log_host_listings_count,2.08,2.4,1.61,1.61,0.2,0.0,0.0
dx_km,2.13,-0.51,0.0,0.0,-0.18,0.0,0.0
log_calc_host_listings_count,1.79,1.97,1.39,1.61,0.15,0.0,0.0
bathrooms_num,1.6,1.45,1.0,1.0,-0.13,0.18,0.16


**Room type and property type.** First table: share of each room type in A/B and in C. Second table: the 15 most common property types of C, whether the model keeps them as their own level (`in_model_top_types`) and whether A/B ever had them (`seen_in_AB`).

In [0]:
rt = pd.concat({"pct_AB": 100 * ab.room_type.value_counts(normalize=True), "pct_C": 100 * c_data.room_type.value_counts(normalize=True)}, axis=1).fillna(0).round(1)
display(rt.rename_axis("room_type").reset_index())

known = set(model.top_types_)
seen_ab = set(ab.property_type.dropna())
pt = c_data.property_type.fillna("(missing)").value_counts().rename("listings").to_frame()
pt["pct_of_C"] = (100 * pt["listings"] / len(c_data)).round(1)
pt["in_model_top_types"] = pt.index.isin(known)
pt["seen_in_AB"] = pt.index.isin(seen_ab)
display(pt.head(15).rename_axis("property_type").reset_index())

share_other = 100 * (~c_data.property_type.isin(known)).mean()
share_unseen = 100 * (~c_data.property_type.isin(seen_ab)).mean()
print(f"listings of C whose property type the model groups as 'Other': {share_other:.1f} %; never seen in A/B: {share_unseen:.1f} %")
if share_unseen > 5:
    note("model", f"{share_unseen:.1f} % of C listings have a property type that A/B never had")

room_type,pct_AB,pct_C
Entire home/apt,71.7,56.1
Private room,26.4,41.8
Hotel room,1.1,0.8
Shared room,0.7,1.3


property_type,listings,pct_of_C,in_model_top_types,seen_in_AB
Private room in home,2042,29.9,true,true
Entire home,1624,23.8,true,true
Entire rental unit,976,14.3,true,true
Entire guesthouse,448,6.6,true,true
Room in hotel,350,5.1,true,true
Entire guest suite,288,4.2,true,true
Entire condo,134,2.0,true,true
Private room in townhouse,126,1.8,true,true
Private room in rental unit,122,1.8,true,true
Entire townhouse,91,1.3,true,true


listings of C whose property type the model groups as 'Other': 1.7 %; never seen in A/B: 0.1 %


**Location.** One row per city: coordinate ranges and the median point (the centre the model measures distances from). Then how far the centre of C is from the training cities, and how many C listings lie inside the coordinate box of A or B.

In [0]:
def box(d):
    return {"lat_min": d.latitude.min(), "lat_max": d.latitude.max(), "lon_min": d.longitude.min(), "lon_max": d.longitude.max(),
            "centre_lat": d.latitude.median(), "centre_lon": d.longitude.median(), "median_dist_to_centre_km": d.dist_to_center_km.median()}

boxes = pd.DataFrame([{"city": c, **box(ab[ab.city == c])} for c in TRAIN_CITIES] + [{"city": CITY_C, **box(c_data)}])
display(boxes.round(3))

dist_rows, inside_any = [], np.zeros(len(c_data), dtype=bool)
c_lat, c_lon = c_data.latitude.median(), c_data.longitude.median()
for c in TRAIN_CITIES:
    b = ab[ab.city == c]
    dy = (c_lat - b.latitude.median()) * 111.0
    dx = (c_lon - b.longitude.median()) * 111.0 * np.cos(np.radians(c_lat))
    dist_rows.append({"from_city": c, "distance_of_C_centre_km": round(float(np.hypot(dx, dy)), 1)})
    inside_any |= (c_data.latitude.between(b.latitude.min(), b.latitude.max()) & c_data.longitude.between(b.longitude.min(), b.longitude.max())).to_numpy()
display(pd.DataFrame(dist_rows))
share_outside = 100 * (~inside_any).mean()
print(f"listings of C outside the coordinate boxes of A and B: {share_outside:.1f} %")
if share_outside > 20 and "latitude" in fitted_columns:   # only matters for a model that uses raw coordinates (v1)
    note("model", f"{share_outside:.1f} % of C lies outside the coordinate boxes of A and B: latitude / longitude are extrapolated by the model")

city,lat_min,lat_max,lon_min,lon_max,centre_lat,centre_lon,median_dist_to_centre_km
Los_Angeles,33.339,34.811,-118.917,-117.654,34.06,-118.341,13.775
San_Francisco,37.709,37.81,-122.512,-122.368,37.778,-122.421,2.55
Santa_Clara_County,36.98,37.464,-122.197,-121.284,37.347,-121.947,8.457


from_city,distance_of_C_centre_km
Los_Angeles,484.1
San_Francisco,63.5


listings of C outside the coordinate boxes of A and B: 100.0 %


**Price level.** One row per city, valid prices only: listings, median, mean, 10th and 90th percentile, share of monthly rentals, and the short-stay median. This is the 'price level the model never saw' check.

In [0]:
def price_row(d, name):
    v = d[pm.valid_price_mask(d)]
    s = v[v.stay_type == "short"]
    return {"city": name, "valid_listings": len(v), "median": v.price.median(), "mean": v.price.mean(), "p10": v.price.quantile(.1),
            "p90": v.price.quantile(.9), "monthly_rentals_pct": 100 * (v.stay_type == "monthly").mean(), "median_short_stay": s.price.median()}

levels = pd.DataFrame([price_row(ab[ab.city == c], c) for c in TRAIN_CITIES] + [price_row(c_data, CITY_C)])
display(levels.round(1))
ab_median = ab[pm.valid_price_mask(ab)].price.median()
ratio_level = levels.iloc[-1]["median"] / ab_median
print(f"median price of C is {ratio_level:.2f} times the A/B median")
if ratio_level > 1.3 or ratio_level < 0.77:
    note("model", f"price level of C differs from A/B: median {levels.iloc[-1]['median']:.0f} against {ab_median:.0f} (ratio {ratio_level:.2f})")

city,valid_listings,median,mean,p10,p90,monthly_rentals_pct,median_short_stay
Los_Angeles,36676,214.5,356.6,69.6,683.5,44.7,284.0
San_Francisco,5676,266.4,428.0,92.8,884.2,35.2,348.5
Santa_Clara_County,5469,180.0,291.8,75.8,525.2,14.3,191.6


median price of C is 0.81 times the A/B median


## 5. Score C with the unchanged saved model

All C listings are scored with the saved model and both saved baselines (fitted on the A/B training rows). The whole city is unseen, so every C listing with a valid price counts for the metrics.

In [0]:
scored_c = pm.score_listings(c_data, model, baseline, baseline_stay, split_label="new_data", model_name=MODEL_NAME,
                             scored_at=f"{datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
c_valid = scored_c[scored_c.error_usd.notna()].copy()
n_nan = int(scored_c.predicted_price.isna().sum())
print(f"scored: {len(scored_c):,} listings, {len(c_valid):,} with a valid price; predictions that are missing: {n_nan}")
if n_nan:
    note("model", f"{n_nan} predictions are missing")
PRED = {"baseline": "baseline_price", "baseline + stay type": "baseline_stay_type_price", "model": "predicted_price"}
results_c = pm.metrics_table(c_valid, PRED, {f"C: {CITY_C}": pd.Series(True, index=c_valid.index)})

scored: 6,831 listings, 5,469 with a valid price; predictions that are missing: 0


**First-run error on C.** One row per model: `mae` = mean absolute error in USD per night, `median_ae` = typical error, `rmse`, `mape_pct` = mean absolute % error, `mean_error` = predicted minus actual (negative = under-prices), `r2_log` = R-squared on log price.

In [0]:
display(results_c.round(2))

scope,model,n,mae,median_ae,rmse,mape_pct,mean_error,r2_log
C: Santa_Clara_County,baseline,5469,132.31,48.0,555.77,36.25,-75.13,0.5
C: Santa_Clara_County,baseline + stay type,5469,130.63,50.46,553.26,39.87,-47.56,0.57
C: Santa_Clara_County,model,5469,122.74,46.48,526.86,37.69,-26.61,0.63


**The three numbers side by side** (assignment 3.7): baseline MAE on C, the model's MAE on A and B (test rows, from the A/B run), and the model's first-run MAE on C. `mape_pct` is added because C has another price level, so dollars alone can mislead.

In [0]:
cols = ["scope", "model", "n", "mae", "median_ae", "rmse", "mape_pct", "mean_error", "r2_log"]
if table_exists(GOLD["metrics"]):
    ab_metrics = read_table(GOLD["metrics"]).filter((F.col("model_name") == MODEL_NAME) & (F.col("split") == "test")).toPandas()
    ab_metrics = ab_metrics[ab_metrics.scope.str.startswith(("A", "B"))][cols]
else:
    ab_metrics = pd.DataFrame(columns=cols)
    note("report", f"{GOLD['metrics']} not found: the A/B numbers are taken from the model's saved metadata (MAE only)")
side = pd.concat([ab_metrics, results_c[cols]], ignore_index=True)

wide = side.pivot(index="scope", columns="model", values="mae")
wide = wide[["baseline", "baseline + stay type", "model"]].reindex([s for s in side.scope.unique() if s in wide.index])
wide["model_vs_baseline_pct"] = (100 * (1 - wide["model"] / wide["baseline"])).round(1)
display(wide.round(2).rename_axis("scope").reset_index())
display(side.round(2))

scope,baseline,baseline + stay type,model,model_vs_baseline_pct
A: Los_Angeles,180.49,171.6,111.52,38.2
B: San_Francisco,245.8,227.33,150.57,38.7
A+B: both,189.25,179.08,116.75,38.3
C: Santa_Clara_County,132.31,130.63,122.74,7.2


scope,model,n,mae,median_ae,rmse,mape_pct,mean_error,r2_log
A: Los_Angeles,baseline,7335,180.49,67.86,579.77,48.43,-90.01,0.55
A: Los_Angeles,baseline + stay type,7335,171.6,57.0,578.68,41.36,-88.59,0.61
A: Los_Angeles,model,7335,111.52,37.07,392.49,26.12,-31.44,0.84
B: San_Francisco,baseline,1136,245.8,84.89,556.05,43.65,-204.25,0.07
B: San_Francisco,baseline + stay type,1136,227.33,69.5,540.04,36.54,-192.45,0.3
B: San_Francisco,model,1136,150.57,53.86,394.11,28.82,-79.6,0.75
A+B: both,baseline,8471,189.25,69.75,576.64,47.79,-105.33,0.49
A+B: both,baseline + stay type,8471,179.08,58.98,573.65,40.72,-102.52,0.58
A+B: both,model,8471,116.75,39.31,392.71,26.48,-37.9,0.83
C: Santa_Clara_County,baseline,5469,132.31,48.0,555.77,36.25,-75.13,0.5


In [0]:
c_row = results_c.set_index("model")
c_base, c_model = c_row.loc["baseline", "mae"], c_row.loc["model", "mae"]
ab_all = side[(side.scope.str.startswith("A+B")) & (side["model"] == "model")]
print(f"Baseline MAE on C:             ${c_base:,.2f}")
print(f"Model {MODEL_NAME} MAE on C:         ${c_model:,.2f}   ({100 * (1 - c_model / c_base):.1f} % {'better' if c_model < c_base else 'WORSE'} than the baseline on C)")
if len(ab_all):
    m_ab = float(ab_all.mae.iloc[0])
    print(f"Model {MODEL_NAME} MAE on A+B (test):   ${m_ab:,.2f}   -> C is {c_model / m_ab:.2f} times the A/B error, {c_model - m_ab:+,.2f} USD")
    if c_model > 1.25 * m_ab:
        note("model", f"the model is {c_model / m_ab:.2f} times worse on C than on A/B (MAE {c_model:.0f} against {m_ab:.0f})")
if c_model >= c_base:
    note("model", "the model does not beat the baseline on C")
bias = c_row.loc["model", "mean_error"]
if abs(bias) > 0.25 * c_valid.actual_price.median():
    note("model", f"systematic bias on C: mean error {bias:.0f} USD per night (negative = under-prices)")

Baseline MAE on C:             $132.31
Model price_model_v2 MAE on C:         $122.74   (7.2 % better than the baseline on C)
Model price_model_v2 MAE on A+B (test):   $116.75   -> C is 1.05 times the A/B error, +5.99 USD


**Sensitivity to extreme prices** (the official numbers above stay as they are). MAE of each model on C with and without the most extreme prices: listings above $1,600, and listings priced above 10 times the A/B median of the same room type and number of guests (`baseline_price`). Shows how much of the error comes from a handful of listings that may be blocking prices. One row per (variant, model).

In [0]:
variants = {
    "all valid prices": pd.Series(True, index=c_valid.index),
    "price up to $1,600": c_valid["actual_price"] <= 1600,
    "price up to 10 x the A/B median of its group": c_valid["actual_price"] <= 10 * c_valid["baseline_price"],
}
sens = pm.metrics_table(c_valid, PRED, variants)
sens["listings_removed"] = sens["scope"].map({k: int((~v).sum()) for k, v in variants.items()})
display(sens[["scope", "model", "n", "listings_removed", "mae", "median_ae", "mape_pct", "mean_error"]].round(2))


scope,model,n,listings_removed,mae,median_ae,mape_pct,mean_error
all valid prices,baseline,5469,0,132.31,48.0,36.25,-75.13
all valid prices,baseline + stay type,5469,0,130.63,50.46,39.87,-47.56
all valid prices,model,5469,0,122.74,46.48,37.69,-26.61
"price up to $1,600",baseline,5403,66,91.93,47.16,35.73,-34.06
"price up to $1,600",baseline + stay type,5403,66,90.42,49.98,39.4,-6.34
"price up to $1,600",model,5403,66,87.36,45.59,37.34,9.83
price up to 10 x the A/B median of its group,baseline,5443,26,107.52,47.86,35.98,-50.07
price up to 10 x the A/B median of its group,baseline + stay type,5443,26,105.96,50.0,39.62,-22.5
price up to 10 x the A/B median of its group,model,5443,26,98.91,46.12,37.44,-2.32


## 6. Where does the error come from?

**By stay type, room type and number of guests.** One row per group: listings and MAE of each model.

In [0]:
c_valid["guests"] = pd.cut(c_valid["accommodates"], [0, 1, 2, 4, 6, 8, 100], labels=["1", "2", "3-4", "5-6", "7-8", "9+"])
for by in (["stay_type"], ["room_type"], ["guests"]):
    print("by", by[0])
    display(pm.mae_by(c_valid, by, PRED).round(2))

by stay_type


stay_type,n,mae_baseline,mae_baseline + stay type,mae_model
monthly,784,79.18,69.49,63.48
short,4671,141.28,140.93,132.66
unknown,14,114.86,115.57,132.61


by room_type


room_type,n,mae_baseline,mae_baseline + stay type,mae_model
Entire home/apt,2929,182.36,179.19,168.18
Hotel room,49,237.41,246.23,200.71
Private room,2403,72.1,72.38,68.94
Shared room,88,51.93,40.61,36.2


by guests


guests,n,mae_baseline,mae_baseline + stay type,mae_model
1,1113,46.63,39.61,39.32
2,1966,69.9,69.27,69.06
3-4,1135,118.54,120.66,114.79
5-6,587,211.43,212.11,200.24
7-8,324,222.32,214.02,213.1
9+,344,591.85,591.04,508.36


**By price band.** One row per band of the actual price: median actual, median predicted and the median of predicted / actual (1 = exact, below 1 = under-priced), plus MAE. Tells whether the whole price level is off or only the tails.

In [0]:
c_valid["actual_price_band"] = pd.cut(c_valid["actual_price"], [0, 100, 200, 400, 800, 1600, 1e9],
                                      labels=["under $100", "$100-200", "$200-400", "$400-800", "$800-1,600", "$1,600 and more"])
c_valid["ratio_pred_to_actual"] = c_valid["predicted_price"] / c_valid["actual_price"]
display(c_valid.groupby("actual_price_band", observed=True).agg(
    listings=("id", "size"), median_actual=("actual_price", "median"), median_predicted=("predicted_price", "median"),
    median_ratio=("ratio_pred_to_actual", "median"), mae=("abs_error_usd", "mean")).round(2).reset_index())

actual_price_band,listings,median_actual,median_predicted,median_ratio,mae
under $100,1204,78.08,98.36,1.29,34.0
$100-200,1820,143.6,161.88,1.14,53.41
$200-400,1517,269.67,286.41,1.05,92.23
$400-800,702,515.45,461.22,0.89,161.16
"$800-1,600",160,1027.5,585.72,0.57,505.14
"$1,600 and more",66,2837.25,824.66,0.34,3018.94


**By known and unknown property type, and by distance from the centre of C.** One row per group: listings and MAE. Shows whether unseen categories or the edges of the county carry the error.

In [0]:
c_valid["property_type_known"] = c_data.loc[c_valid.index, "property_type"].isin(known)
c_valid["distance_band"] = pd.cut(c_data.loc[c_valid.index, "dist_to_center_km"], [-1, 5, 10, 20, 40, 1e6],
                                  labels=["0-5 km", "5-10 km", "10-20 km", "20-40 km", "over 40 km"])
for by in (["property_type_known"], ["distance_band"]):
    print("by", by[0])
    display(pm.mae_by(c_valid, by, PRED).round(2))

by property_type_known


property_type_known,n,mae_baseline,mae_baseline + stay type,mae_model
false,99,118.22,115.81,124.71
true,5370,132.57,130.9,122.71


by distance_band


distance_band,n,mae_baseline,mae_baseline + stay type,mae_model
0-5 km,1145,101.46,99.2,95.55
5-10 km,2094,98.7,98.72,102.58
10-20 km,1897,164.24,162.44,139.65
20-40 km,273,275.16,262.24,252.5
over 40 km,60,234.23,239.13,220.11


**Listings the model under-prices most** and **over-prices most** (12 each).

In [0]:
COLS = ["id", "room_type", "property_type", "accommodates", "bedrooms", "minimum_nights", "actual_price", "predicted_price", "error_usd", "pct_error"]
print("under-priced"); display(c_valid.nsmallest(12, "error_usd")[COLS])
print("over-priced"); display(c_valid.nlargest(12, "error_usd")[COLS])

under-priced


id,room_type,property_type,accommodates,bedrooms,minimum_nights,actual_price,predicted_price,error_usd,pct_error
53928787,Entire home/apt,Entire home,16.0,1.0,7.0,19042.86,762.74,-18280.12,-95.99
1604494149101566186,Entire home/apt,Entire home,16.0,7.0,3.0,17677.67,2240.32,-15437.35,-87.33
1468057604214860595,Entire home/apt,Entire rental unit,6.0,2.0,1.0,10122.98,318.65,-9804.33,-96.85
976051093982679390,Entire home/apt,Entire home,4.0,4.0,1.0,9776.0,954.11,-8821.89,-90.24
1467953841868690662,Entire home/apt,Entire rental unit,6.0,2.0,1.0,8999.1,313.65,-8685.45,-96.51
54241467,Entire home/apt,Entire villa,6.0,3.0,7.0,9098.29,539.89,-8558.4,-94.07
1390085402346093379,Entire home/apt,Entire home,10.0,3.0,1.0,9176.85,820.96,-8355.89,-91.05
842723134393928713,Entire home/apt,Entire home,8.0,4.0,2.0,8206.0,574.56,-7631.44,-93.0
1717658760978833089,Entire home/apt,Entire home,14.0,5.0,31.0,7565.72,738.59,-6827.13,-90.24
1561045247406309186,Entire home/apt,Entire home,9.0,4.0,3.0,7940.33,1118.58,-6821.75,-85.91


over-priced


id,room_type,property_type,accommodates,bedrooms,minimum_nights,actual_price,predicted_price,error_usd,pct_error
1258804744420284014,Private room,Room in hotel,15.0,9.0,1.0,812.0,2702.95,1890.95,232.88
1685751029894051580,Entire home/apt,Entire home,10.0,5.0,1.0,1245.0,2501.6,1256.6,100.93
1571039708758258861,Entire home/apt,Entire villa,10.0,5.0,1.0,699.0,1822.82,1123.82,160.78
1239237683991806613,Hotel room,Room in hotel,16.0,4.0,1.0,1075.0,2171.12,1096.12,101.97
1508163171007055892,Hotel room,Room in hotel,12.0,3.0,1.0,806.4,1888.95,1082.55,134.25
1508661750693434684,Hotel room,Room in hotel,16.0,4.0,1.0,1152.0,2202.64,1050.64,91.2
1592573062064783844,Entire home/apt,Entire home,16.0,5.0,3.0,498.0,1443.14,945.14,189.79
668909950740206887,Entire home/apt,Entire villa,10.0,5.0,28.0,762.28,1675.75,913.47,119.83
1258772571955469771,Private room,Room in hotel,12.0,4.0,1.0,584.0,1480.05,896.05,153.43
1258801456360308007,Private room,Room in hotel,10.0,6.0,1.0,541.0,1425.94,884.94,163.58


## 7. Observations from this first run

Everything noted above, in one table. This is the raw material for step 8: each item becomes either a **pipeline fix** (the pipeline should have handled it from the start: an unexpected value, a schema difference, a failed download) or a **model change** (the model did not transfer: a feature that means something else in C, a price level it never saw).

In [0]:
obs = pd.DataFrame(OBSERVATIONS, columns=["area", "observation"])
obs.insert(0, "#", range(1, len(obs) + 1))
display(obs if len(obs) else pd.DataFrame({"observation": ["nothing flagged automatically: check the tables above"]}))

observation
nothing flagged automatically: check the tables above


## 8. Publish the first run

The C rows of the first run are added to `gold_predictions` (model `MODEL_NAME`, split `new_data`) and the C metrics to the metrics table; **rows of A/B and of other model versions are not touched**. The small result files go to `results/` so they can be committed.

In [0]:
if PUBLISH:
    gold_c = scored_c.astype(object).where(scored_c.notna(), None)
    write_rows(spark.createDataFrame(gold_c, schema=pm.GOLD_DDL), GOLD["predictions"],
               f"model_name = '{MODEL_NAME}' AND city IN ({sql_list(scored_c.city)})")
    metrics_c = results_c.assign(split="new_data", model_name=MODEL_NAME, evaluated_at=f"{datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
    write_rows(spark.createDataFrame(metrics_c), GOLD["metrics"],
               f"model_name = '{MODEL_NAME}' AND scope IN ({sql_list(results_c.scope)})")
    n_gold = read_table(GOLD["predictions"]).filter((F.col("model_name") == MODEL_NAME) & (F.col("city") == CITY_C)).count()
    assert n_gold == len(scored_c), f"gold table has {n_gold} rows for C, expected {len(scored_c)}"
    print(f"published: {n_gold:,} rows of city C in {GOLD['predictions']}")
else:
    print("PUBLISH is False: nothing written")

os.makedirs(RESULTS_DIR, exist_ok=True)
side.round(3).to_csv(f"{RESULTS_DIR}/{MODEL_NAME}_city_c_first_run.csv", index=False)
with open(f"{RESULTS_DIR}/{MODEL_NAME}_city_c_first_run.md", "w") as f:
    f.write(f"City C first run: {MODEL_NAME}, source of C: {source_used}\n\n")
    f.write(pm.metrics_to_markdown(side, {"model_name": MODEL_NAME, "chosen_candidate": meta.get("chosen_candidate"), "trained_at": meta.get("trained_at")}))
    f.write("\nObservations\n\n" + "\n".join(f"- [{a}] {t}" for a, t in OBSERVATIONS) + "\n")
print("written to", RESULTS_DIR)

published: 6,831 rows of city C in workspace.airbnb.gold_predictions
written to /Workspace/Repos/bobas.pn@ucu.edu.ua/PizzaByte_group_assignment_2/results


## 9. What comes next (step 8)
1. Turn the observations into the change list, one row each, marked **pipeline fix** or **model change**, with the evidence (the table that shows it).
2. Pipeline fixes belong to the 3.3 pipeline (rerun it for C, then rerun this notebook: it is the same unchanged v1 on cleaner data, so keep the v1 numbers of the first run for the report).
3. Model changes: edit `FEATURES`, `CANDIDATES` or `add_features` in `src/price_model.py`, set `MODEL_NAME = "price_model_v2"` in `price_model.ipynb` and run it (the stored A/B split is reused and a new model is fitted; its rows are written next to v1 because `model_name` differs). A model fitted earlier always scores with the columns it was fitted on, so v1 stays reproducible.
4. Score C with v2 by changing `MODEL_NAME` here and running again: its C rows land in `gold_predictions` next to v1.
5. Report (3.7): baseline MAE on C, A/B model MAE, and C's MAE for v1 and v2, with the reasons for the gap.